# Day 94: Teach the alarm to ignore tiny groups (`min_n`)

**Goal:** on Day 92 the gap "25 to 57 times" rested on a dataset with **one layer**. Today the function itself refuses tiny groups, so a single layer can never decide the answer again.

**Six small steps:** 1. why small groups mislead · 2. `.size()` and `.count()` · 3. the sieve · 4. `min_n` in the function · 5. guards and tests · 6. the real soil data.

> Restart the kernel, then run top to bottom.

## 0. Setup
`agronomic_functions.py` and `db_config.py` live two folders up, so I add that folder to the search path **before** importing.

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
from agronomic_functions import build_flat, quality_flags, dataset_alarm
from db_config import run_query

## Step 1: why small groups mislead
**Picture:** one restaurant has 500 reviews averaging 4 stars, another has 1 review with 5 stars. I trust the 500. One review might be from one lucky day.

- With 900 values, one strange value barely moves the median. It is **stable**.
- With 1 value, the median **is** that value.

**In my project:** `WD-NWAFU-SCS` has 1 layer. Its median (1.8) is just that layer, and it made the gap look like 57 times instead of 25 to 43.

**Rule:** before comparing groups, check how many values each has, and leave out groups that are too small. The smallest group size I accept is **`min_n`** (minimum number). I use 10. It is a choice, not a law, so it is a setting I can change.

**One sentence to teach it:** *Don't judge a group by one member.*

**My check answers:**
1. Two layers, 3 and 300: the median is (3 + 300) / 2 = **151.5**. No real layer is anywhere near 151.5, and one more layer could swing it to almost anything, so it is not "typical".
2. I trust Group A (500 layers, median 4.0). Group B's median is just its one layer (90).
3. `min_n`: groups with fewer than `min_n` values are ignored by the alarm, so they cannot mess up the comparison.

## Step 2: two ways to count a group
**Picture:** a class register has 30 names (`size`), but only 25 students handed in homework (`count`).
- `.size()` counts **every row** in the group.
- `.count()` counts only rows with a **real value** in that column. Missing values are skipped.

A median uses only real values, so **`min_n` must use `.count()`**. In my data, GH-GhaSP has 935 rows (`size`) but 0 clay values (`count`).

In [ ]:
df = pd.DataFrame({
    "dataset": ["A", "A", "A", "B", "B"],
    "clay":    [30, None, 25, 40, 35]
})
print(df.groupby("dataset").size())            # A = 3, B = 2   (all rows)
print(df.groupby("dataset")["clay"].count())   # A = 2, B = 2   (only real values)

**Check.** Prediction: `size` X = 3, Y = 1; `count` X = 1, Y = 1. With `min_n = 2`, `size` lets X pass but `count` lets nobody pass. `count` is the honest one: one real value cannot give a trustworthy median.

In [ ]:
t = pd.DataFrame({
    "g":  ["X", "X", "X", "Y"],
    "ph": [6.0, None, None, 5.5]
})
print(t.groupby("g").size())
print(t.groupby("g")["ph"].count())

### Look-alikes (for my cheat sheet)
| Command | What it gives | Use it when |
|---|---|---|
| `len(df)` | number of rows | "How big is my table?" |
| `df.shape` | (rows, columns) | checking a table after a join |
| `df.size` (no brackets) | rows × columns (every cell) | rarely; a trap, it is not the number of rows |
| `groupby(...).size()` | every row in each group | "How many rows per group?" |
| `groupby(...)[col].count()` | only **real** values per group | "Do I have enough data?" (`min_n`) |
| `.isna().sum()` | how many are **missing** | cleaning reports |
| `(condition).sum()` | how many rows are **True** | counting flags |
| `.value_counts()` | how often each value appears (hides missing unless `dropna=False`) | "How many layers per dataset?" |
| `.nunique()` | how many **different** values | "How many datasets are there?" |

## Step 3: the sieve
**Picture:** pour all the groups in. Big ones pass through, tiny ones stay behind. Three moves:
1. ask each group a yes/no question: `counts >= min_n`
2. keep only the "yes" **names**: `counts[counts >= min_n].index`
3. keep only those groups' medians: `medians[medians.index.isin(enough)]`

**Prediction:** counts A = 4, B = 3 (the `None` does not count), C = 4, D = 1. With `min_n = 3`, A, B and C pass. With `min_n = 4`, only A and C pass.

In [ ]:
toy = pd.DataFrame({
    "g": ["A"]*4 + ["B"]*4 + ["C"]*4 + ["D"],
    "v": [4, 5, 6, 5,   5, 6, None, 6,   4, 5, 6, 5,   300]
})
counts  = toy.groupby("g")["v"].count()
medians = toy.groupby("g")["v"].median()

print(counts >= 4)                         # Move 1: a yes/no for each group
enough = counts[counts >= 4].index         # Move 2: the names that pass
print(enough)
print(medians[medians.index.isin(enough)]) # Move 3: A and C only

## Step 4: `min_n` in the function
The sieve goes **straight after the medians**, so tiny groups are removed **before** the typical value is worked out. Four changes:
1. a new setting `min_n=10` in the `def` line
2. a guard: `min_n` below 1 makes no sense (a group cannot have a negative number of values, and 0 would let every group pass)
3. the sieve (the three moves above)
4. a guard: if **no** group passes, stop with a clear message

**Why the last guard matters:** if every group is removed, `medians` is empty and `typical` becomes `NaN`. `NaN == 0` is `False`, so a guard that only checks `== 0` would not stop it, and the function would quietly return an empty result that looks like "nothing odd". The new message says exactly why it stopped.

Reference copy (the real one lives in `csa_soil_project/agronomic_functions.py`):
```python
def dataset_alarm(df, column, group="dataset_id", ratio=5, min_n=10):
    """Finds groups whose median is far above the typical median.

    Groups with fewer than min_n real values are left out before the typical
    value is found, preventing tiny sample sizes from creating false alarms.

    Args:
        df: The pandas DataFrame.
        column: The numerical column to check.
        group: The column to group by (default is "dataset_id").
        ratio: The multiplier threshold to trigger a flag (default is 5).
        min_n: The minimum number of real values a group must have to be
               included in the comparison (default is 10).

    Returns:
        A pandas Series containing the medians of the flagged groups.

    Raises:
        ValueError: If the column or group doesn't exist in the table.
        ValueError: If min_n is less than 1.
        ValueError: If no group has at least min_n values.
        ValueError: If the typical value is 0 or missing.

    Note:
        Good for values where "times bigger" means something (carbon, clay %).
        Weak for pH, which is a log scale.

    Example:
        dataset_alarm(traced, "orgc")   # flags GH-GhaSP
    """
    if column not in df.columns:
        raise ValueError(f"Column {column} not found in table.")
    if group not in df.columns:
        raise ValueError(f"Group {group} not found in table.")
    if min_n < 1:
        raise ValueError("min_n must be at least 1.")

    counts = df.groupby(group)[column].count()
    enough = counts[counts >= min_n].index
    medians = df.groupby(group)[column].median()
    medians = medians[medians.index.isin(enough)]      # the sieve

    if len(medians) == 0:
        raise ValueError(f"No {group} has at least {min_n} values in {column}.")

    typical = medians.median()
    if pd.isna(typical) or typical == 0:
        raise ValueError("Typical value is 0 or missing. Cannot calculate ratio.")

    ratios = medians / typical
    return medians[ratios > ratio]
```
**Docstring lesson:** "outlier" would promise more than the code does. It only flags groups far **above** typical, so the docstring says "far above".

In [ ]:
help(dataset_alarm)

## Step 5: the tests
**5a. A changed default breaks old tests, as it should.** My old toy table has 3 layers per dataset. With the new default `min_n=10`, no group passes, so the function raised `No dataset_id has at least 10 values in value.` That is the test doing its job.

**5b. One thing at a time.** The old checks test the **comparison**, not group size, so they say `min_n=3`. The typical-is-0 check needs `min_n=3` too: without it, it would still pass, but because of the **sieve** guard, not the zero guard. **A test can pass for the wrong reason.**

**5c. New checks for the sieve,** on the table where D is one strange layer:
- `min_n=1`: medians A 5, B 6, C 5, D 300; typical 5.5; D is 54.5 times typical, so **D is flagged** (a false alarm from one layer)
- `min_n=3`: D is sieved out; typical 5; ratios 1.0, 1.2 and 1.0, so **nothing is flagged**

**5d. Proof the new test can fail:** with the sieve line switched off (`#`), check 5 failed with *"Nothing should be flagged when min_n=3"*. With it back on, all tests passed. Only check 5 noticed, so each new feature needs **its own** check.

In [ ]:
def test_dataset_alarm():
    toy = pd.DataFrame({
        "dataset_id": ["A", "A", "A", "B", "B", "B", "C", "C", "C"],
        "value":      [4, 5, 6,   5, 6, 7,   100, 110, 120]
    })

    # Check 1: only C is flagged (small groups allowed for this test)
    result = dataset_alarm(toy, "value", min_n=3)
    assert list(result.index) == ["C"], "C should be the only flagged dataset"

    # Check 2: a very high ratio flags nothing
    result_high = dataset_alarm(toy, "value", ratio=50, min_n=3)
    assert len(result_high) == 0, "Nothing should be flagged at ratio 50"

    # Check 3: a missing column raises ValueError
    try:
        dataset_alarm(toy, "wrong_column")
        raise AssertionError("Should have raised ValueError for a missing column")
    except ValueError:
        pass

    # Check 3b: a typical value of 0 raises ValueError (min_n=3, so the ZERO guard fires)
    zero = pd.DataFrame({
        "dataset_id": ["A", "A", "A", "B", "B", "B", "C", "C", "C"],
        "value":      [0, 0, 0,   0, 0, 0,   5, 6, 7]
    })
    try:
        dataset_alarm(zero, "value", min_n=3)
        raise AssertionError("Should have raised ValueError when typical is 0")
    except ValueError:
        pass

    small = pd.DataFrame({
        "g": ["A"]*4 + ["B"]*4 + ["C"]*4 + ["D"],
        "v": [4, 5, 6, 5,   5, 6, None, 6,   4, 5, 6, 5,   300]
    })

    # Check 4: a tiny group decides the answer if you let it (min_n=1)
    result_tiny = dataset_alarm(small, "v", group="g", min_n=1)
    assert list(result_tiny.index) == ["D"], "D should be flagged when min_n=1"

    # Check 5: the sieve stops it (min_n=3)
    result_sieve = dataset_alarm(small, "v", group="g", min_n=3)
    assert len(result_sieve) == 0, "Nothing should be flagged when min_n=3"

    # Check 6: a min_n that makes no sense
    try:
        dataset_alarm(small, "v", group="g", min_n=0)
        raise AssertionError("Did not raise ValueError for min_n=0")
    except ValueError:
        pass

    # Check 7: nobody passes the sieve
    try:
        dataset_alarm(small, "v", group="g", min_n=10)
        raise AssertionError("Did not raise ValueError when no group has enough values")
    except ValueError:
        pass

    print("all tests passed")

test_dataset_alarm()

## Step 6: the real soil data

In [ ]:
flat = build_flat("orgc", ["ph", "clay", "sand"])
checked = quality_flags(flat)
meta = run_query("SELECT layer_id, dataset_id FROM wosis_orgc WHERE country_name = 'Ghana'")
traced = pd.merge(checked, meta, on="layer_id", how="left")
print(len(traced))        # must be 1756

**Predictions** (layers per dataset: AF-AfSP 722, GH-GhaSP 935, WD-WISE 79, WD-ISIS 19, WD-NWAFU-SCS 1):
1. For `orgc`, the sieve removes **WD-NWAFU-SCS** (1 layer).
2. New typical value: remaining medians 2.4, 3.2, 4.2, 103.4, so (3.2 + 4.2) / 2 = **3.7**.
3. **GH-GhaSP is still flagged**, about 103.4 / 3.7 ≈ **28 times** typical.
4. For `clay`, **AF-AfSP, WD-ISIS and WD-WISE** pass. GH-GhaSP and WD-NWAFU-SCS have 0 clay values.

In [ ]:
counts_orgc = traced.groupby("dataset_id")["orgc"].count()
print(counts_orgc)                       # WD-NWAFU-SCS has 1, so it is sieved out

for col in ["orgc", "ph", "clay", "sand"]:
    print(col, "->", dataset_alarm(traced, col).to_dict())

**What I found:** `orgc -> {'GH-GhaSP': 103.4}`, and `{}` for pH, clay and sand. The **typical value moved** (the single layer no longer counts), but the **finding stayed the same**. That is what a good fix does: the answer gets more honest without becoming a different answer.

**The script still works without any change** (it uses the new default). Expected: the same report as before and exit code 0.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "../../run_checks.py"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("exit code:", result.returncode)

## What I keep from today
- **Don't judge a group by one member.** A tiny group can create a false alarm.
- **`.count()`, not `.size()`,** answers "is there enough data?", because a median uses only real values.
- **The sieve** removes tiny groups **before** the typical value is found.
- **Guards stop silent wrong answers:** an empty result that looks like "all fine" is worse than an error.
- **A changed default should break old tests.** That is the test doing its job.
- **A test can pass for the wrong reason.** Make it fail once on purpose, and give each new feature its own check.
- **A docstring must not promise more than the code does** ("far above", not "outlier").

**Next:** Days 95 to 98, exploratory statistics (curriculum Week 14).